# 03 · The filesystem middleware

In notebook 01 we built an agent with no tools and it had nine. Seven of those
came from one layer:

`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`, `execute`

That is not a random selection. It is, almost exactly, what an engineer does at
a terminal.

This notebook is about why giving a model that particular set changes how it
behaves — and about one feature of this middleware that quietly solves the
hardest problem in agent design.

In [1]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

repo root: /Users/aseem/Documents/hubbleflow/standalone-projects/agentic-crew


## A filesystem is a place to think

Give a model only a chat history and everything it knows has to fit in its
context window. It re-derives the same conclusions, forgets what it decided
four turns ago, and gets more expensive with every message.

Give it a filesystem and something changes. It can write a plan down, work
through it, and come back. Notes outlive the turn that produced them. And
critically — another agent can read them.

That last point is why the crew shares one volume. A backend engineer does not
message the reviewer a diff. It writes files, and the reviewer reads them.

In [2]:
from deepagents.backends import FilesystemBackend
import inspect

print(inspect.signature(FilesystemBackend.__init__))
print()
print("Our agents are constructed with:")
src = (ROOT / "agents/shared/agent_loop.py").read_text()
start = src.index("def _filesystem_backend")
print(src[start:src.index("def _skill_sources")].rstrip())

(self, root_dir: str | pathlib._local.Path | None = None, virtual_mode: bool = True, max_file_size_mb: int = 10) -> None

Our agents are constructed with:
def _filesystem_backend() -> FilesystemBackend:
    """The project volume, as the agent's filesystem.

    ``virtual_mode`` roots every path the model uses at this directory, so a
    model that asks for ``/etc/passwd`` gets ``<workspace>/etc/passwd``.
    """
    Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
    return FilesystemBackend(root_dir=WORKSPACE, virtual_mode=True)


`virtual_mode=True` is doing real work there. Every path the model uses is
rooted at that directory, so a model that asks for `/etc/passwd` gets
`<workspace>/etc/passwd`. It is not a policy the model is asked to respect; it
is arithmetic on the path before the read happens.

And in the cluster, the volume is mounted with `subPath: <project-id>`, so one
project's `/workspace` is a different directory on disk from another's. Two
layers, both structural.

## Watch it work

A real agent, a real directory, no instructions about *how* to use files.

In [3]:
import os, tempfile
from deepagents import create_deep_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage

assert os.environ.get("GOOGLE_API_KEY"), "set GOOGLE_API_KEY (see .env)"

workdir = tempfile.mkdtemp(prefix="nb03-")

agent = create_deep_agent(
    model=ChatGoogleGenerativeAI(model="gemini-3.5-flash"),
    system_prompt="You are a backend engineer. Work in files, not in your reply.",
    backend=FilesystemBackend(root_dir=workdir, virtual_mode=True),
)

result = await agent.ainvoke({"messages": [{"role": "user", "content":
    "Write a Python function `is_healthy(pool)` that returns False when the pool "
    "is exhausted, into src/health.py. Then write one test for it in tests/test_health.py."
}]})

print("Tool calls the model chose to make:")
for m in result["messages"]:
    if isinstance(m, AIMessage):
        for tc in (m.tool_calls or []):
            arg = tc["args"].get("file_path") or tc["args"].get("path") or ""
            print(f"   {tc['name']:<12} {arg}")

Tool calls the model chose to make:
   glob         
   ls           /
   glob         
   glob         
   ls           /workspace
   ls           /app
   ls           /home
   ls           /tmp
   write_file   /src/health.py
   ls           /src
   write_file   /src/health.py
   write_file   /tests/test_health.py
   glob         
   read_file    /src/health.py
   read_file    /tests/test_health.py


In [4]:
for path in sorted(pathlib.Path(workdir).rglob("*")):
    if path.is_file():
        print("=" * 60)
        print(path.relative_to(workdir))
        print("=" * 60)
        print(path.read_text())

src/health.py
def is_healthy(pool) -> bool:
    """
    Determines if the given pool is healthy.
    Returns False when the pool is exhausted, and True otherwise.

    A pool is considered exhausted if:
    - It has an 'is_exhausted' attribute or method that is True/returns True.
    - It has an 'exhausted' attribute or method that is True/returns True.
    - It has an 'empty' attribute or method (meaning no resources are available) that is True/returns True.
    """
    if pool is None:
        return False

    # Check 'is_exhausted'
    if hasattr(pool, 'is_exhausted'):
        is_ex = pool.is_exhausted
        if callable(is_ex):
            if is_ex():
                return False
        elif is_ex:
            return False

    # Check 'exhausted'
    if hasattr(pool, 'exhausted'):
        ex = pool.exhausted
        if callable(ex):
            if ex():
                return False
        elif ex:
            return False

    # Check 'empty' (e.g. pool of available connection

Nobody told it to create `src/` and `tests/`. It behaves like an engineer
because it was handed an engineer's tools, and the conventions came with them.

This is the cheapest lever in the whole harness. Changing what an agent *is*
mostly means changing what it can touch.

## The part that matters most: eviction

Here is the problem every agent system hits.

An agent runs `pytest` on a large suite. The output is 40,000 tokens. That
result goes into the message history, and now every subsequent model call
carries it. Two more commands like that and the context window is full of
console output, the useful conversation has been squeezed out, and the agent
starts forgetting what it was doing.

The usual fix is to write `command > out.txt` and read back only what you need.
`FilesystemMiddleware` does that for you.

In [5]:
from deepagents import FilesystemMiddleware
import inspect

sig = inspect.signature(FilesystemMiddleware.__init__)
for name, param in sig.parameters.items():
    if "token" in name or "evict" in name:
        print(f"  {name} = {param.default}")

  tool_token_limit_before_evict = 20000
  human_message_token_limit_before_evict = 50000


A tool result over `tool_token_limit_before_evict` is **written to the
filesystem and replaced in the history with its path**. The agent sees
something like "output was large, it is at `/tmp/tool-output-3.txt`" and can
`grep` it for the three lines it actually wanted.

`human_message_token_limit_before_evict` does the same for enormous pasted
input.

Two consequences worth stating plainly:

* **A long-running agent stops being a context-window problem.** Its working
  memory is disk; its context holds the pointer.
* **This is why the filesystem is required.** In `deepagents`,
  `FilesystemMiddleware` is not optional — you can replace the backend, not
  remove the layer. Eviction has to have somewhere to put things.

In [6]:
import deepagents.graph as g
print("Middleware the library will not let you remove:")
for cls, _aliases in g._REQUIRED_MIDDLEWARE:
    print("   ", cls.__name__)

Middleware the library will not let you remove:
    FilesystemMiddleware
    SubAgentMiddleware


## What you now know

1. The filesystem tools make an agent behave like an engineer, without being
   told to.
2. `virtual_mode` roots every path structurally; `subPath` does it again at the
   volume. Neither relies on the model cooperating.
3. **Automatic eviction** turns large tool output into a file path, so context
   exhaustion stops being the limiting factor on how long an agent can work.
4. The filesystem layer is required, because eviction depends on it.

Next: instructions are files too. `SkillsMiddleware`, and why the crew's
knowledge is not in its system prompt.